# Set ALL Your API Path

In [21]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR GOOGLE API KEY"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR HUGGINGFACE API TOKEN"

## Import ALL The Necessary Library Needed

In [ ]:
import os
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores.faiss import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
from unstructured.partition.pdf import partition_pdf

##### Stored The Downoaded 10k-filling pdf in Directory FILING_DIRECTORY

##### Store the Faiss Vector Embedding in the Directory VECTOR_STORE_PATH

In [10]:
FILINGS_DIR = "10k_filings" 
VECTOR_STORE_PATH = "faiss_index_pdf"

# 1. Data Loading and Processing

In [ ]:
def load_and_clean_pdfs():
    """
    Loads and cleans PDF documents from the filings directory using `unstructured`.
    This approach is effective at extracting text from complex layouts and tables.
    """
    documents = []
    print(f"--- 1. Loading and cleaning PDF documents from '{FILINGS_DIR}' ---")
    
    for filename in os.listdir(FILINGS_DIR):
        if filename.lower().endswith(".pdf"):
            filepath = os.path.join(FILINGS_DIR, filename)
            try:
                # Use the "fast" strategy to avoid the need for external dependencies like poppler.
                elements = partition_pdf(
                    filename=filepath,
                    strategy="fast"
                )
                
                # Combine the text from all detected elements into a single string
                text = "\n\n".join([el.text for el in elements])

                # Extract metadata from the filename (e.g., "GOOGL_2024_10-K.pdf")
                base_filename = os.path.splitext(filename)[0]
                parts = base_filename.replace('_10-K', '').split('_')
                company, year = parts[0], parts[1]
                
                documents.append({
                    "content": text,
                    "metadata": {"company": company, "year": year, "source": filename}
                })
                print(f"  Successfully processed {filename}")
            except Exception as e:
                print(f"!!! ERROR processing {filename}: {e}")
    
    print(f"--- 2. Loaded and cleaned {len(documents)} PDF documents. ---")
    return documents

# 2. Vector Store Creation

In [ ]:

def create_and_save_vector_store():
    """Creates a FAISS vector store from the cleaned PDF text."""
    if os.path.exists(VECTOR_STORE_PATH):
        print(f"Vector store already exists at '{VECTOR_STORE_PATH}'. Skipping creation.")
        return

    docs = load_and_clean_pdfs()
    if not docs:
        print("No documents found. Please ensure your PDF files are in the '10k_filings' folder.")
        return

    print("--- 3. Chunking documents into smaller pieces... ---")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    texts = [doc['content'] for doc in docs]
    metadatas = [doc['metadata'] for doc in docs]
    chunks = text_splitter.create_documents(texts, metadatas=metadatas)
    
    print(f"--- 4. Created {len(chunks)} text chunks. ---")
    if not chunks: return

    print("--- 5. Initializing local Hugging Face embedding model... ---")
    # This model runs on your machine and will be downloaded on the first run.
    embeddings_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    print("--- 6. Generating embeddings and creating vector store (this may take a while)... ---")
    vector_store = FAISS.from_documents(chunks, embeddings_model)
            
    vector_store.save_local(VECTOR_STORE_PATH)
    print(f"--- 7. Vector store saved successfully to '{VECTOR_STORE_PATH}'. ---")


#### Run Code to Start The Embedding if Embedded Data is Not Present

In [13]:
create_and_save_vector_store()

Vector store already exists at 'faiss_index_pdf'. Skipping creation.


# Defining Class For FinancialQASystem

In [18]:
class FinancialQASystem:
    """The core agentic system for handling financial queries."""
    def __init__(self):
        print("Initializing Financial Q&A System...")
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)
        self.embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.vector_store = FAISS.load_local(VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True)
        self.retriever = self.vector_store.as_retriever(search_kwargs={"k": 5})
        self._setup_prompts()
        print("System initialized successfully.")
    
    def _setup_prompts(self):
        """Defines the prompt templates that guide the LLM."""
        self.decomposition_prompt = PromptTemplate.from_template(
            """
            Analyze the user's financial query. If it's complex (involves comparisons, calculations, or multiple entities/periods), break it down into a list of simpler sub-queries. Otherwise, return the original query.
            Return a JSON object with a single key "sub_queries" containing the list of strings.
            Query: "{query}"
            JSON Output:
            """
        )
        self.synthesis_prompt = PromptTemplate.from_template(
            """
            You are a meticulous financial analyst. Based ONLY on the provided Context from 10-K filings, answer the Original Query.
            Your response must be a single JSON object with the following keys: "query", "answer", "reasoning", "sub_queries", and "sources".
            - "answer": A direct, concise answer. If the information is not in the context, state that clearly.
            - "reasoning": A brief, step-by-step explanation of how you derived the answer from the context.
            - "sources": A list of objects, each containing "company", "year", and a direct "excerpt" from the context that supports the answer.
            
            Original Query: {query}
            Sub-Queries Used: {sub_queries}
            Context:
            ---
            {context}
            ---
            JSON Response:
            """
        )

    def _decompose_query(self, query: str) -> list[str]:
        """Uses the LLM to break down a complex query into sub-queries."""
        print(f"\n> Decomposing query: '{query}'")
        chain = self.decomposition_prompt | self.llm | StrOutputParser()
        response_str = chain.invoke({"query": query})
        try:
            json_str = response_str.strip().replace("```json", "").replace("```", "").strip()
            result = json.loads(json_str)
            return result.get("sub_queries", [query])
        except json.JSONDecodeError:
            print("  - Warning: Failed to parse decomposition. Using original query.")
            return [query]

    def _retrieve_context(self, sub_queries: list[str]) -> str:
        """Retrieves relevant document chunks for a list of sub-queries."""
        print("> Retrieving context...")
        all_docs = []
        for sub_query in sub_queries:
            all_docs.extend(self.retriever.invoke(sub_query))
        unique_docs = {doc.page_content: doc for doc in all_docs}.values()
        print(f"  - Retrieved {len(unique_docs)} unique document chunks.")
        return "\n\n".join([f"Source: {doc.metadata['source']}\nContent: {doc.page_content}" for doc in unique_docs])

    def answer_query(self, query: str):
        """Orchestrates the full process of answering a query."""
        sub_queries = self._decompose_query(query)
        context = self._retrieve_context(sub_queries)
        print("> Synthesizing final answer...")
        synthesis_chain = self.synthesis_prompt | self.llm | StrOutputParser()
        response_str = synthesis_chain.invoke({
            "query": query, "context": context, "sub_queries": str(sub_queries)
        })
        try:
            json_str = response_str.strip().replace("```json", "").replace("```", "").strip()
            return json.loads(json_str)
        except json.JSONDecodeError:
            return {"error": "Could not generate valid JSON response.", "raw_output": response_str}

# Ask The Question and If done Type exit or quit

In [19]:
def main():
    """Main function to run the application."""
    create_and_save_vector_store()
    qa_system = FinancialQASystem()
    
    print("\n--- 🤖 Financial Q&A System is Ready ---")
    print("Type 'quit' or 'exit' to end the session.")
    
    while True:
        user_query = input("\nEnter your financial question: ")
        if user_query.lower() in ['quit', 'exit']: break
        response = qa_system.answer_query(user_query)
        print("\n--- Formatted JSON Response ---")
        print(json.dumps(response, indent=2))

# Start QASystem

In [20]:
main()

Vector store already exists at 'faiss_index_pdf'. Skipping creation.
Initializing Financial Q&A System...
System initialized successfully.

--- 🤖 Financial Q&A System is Ready ---
Type 'quit' or 'exit' to end the session.

> Decomposing query: 'What was NVIDIA's total revenue in fiscal year 2024?'
> Retrieving context...
  - Retrieved 5 unique document chunks.
> Synthesizing final answer...

--- Formatted JSON Response ---
{
  "query": "What was NVIDIA's total revenue in fiscal year 2024?",
  "answer": "NVIDIA's total revenue in fiscal year 2024 was $60,922 million.",
  "reasoning": "The provided context from NVIDIA's 2024 10-K filing includes a table summarizing revenue by specialized markets, with a row for \"Total revenue\" and a column for \"Jan 28, 2024\" (which represents the end of fiscal year 2024). The value in that cell is $60,922 million.",
  "sub_queries": [
    "What was NVIDIA's total revenue in fiscal year 2024?"
  ],
  "sources": [
    {
      "company": "NVDA",
      "